# Optimize the 2050 energy system with varying constraints on burden shifts

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path
import pickle

In [ ]:
save_results = False
run_model = False
update_dat_files = False

In [ ]:
NOTEBOOK_DIR = Path.cwd()
LCA_PROJECTS_ROOT = NOTEBOOK_DIR.parent.parent
sys.path.insert(1, str(LCA_PROJECTS_ROOT / '00_Shared'))

In [ ]:
from utils import (
    run_opti,
    run_opti_multi_iam,
    add_biogenic_climate_change_to_impact_scores_df,
    add_rhhd_and_reqd_to_impact_scores_df,
    get_impact_scores,
    category_to_sector,
    aggregate_mobility_submodels,
    plot_configuration_sector,
    plot_contribution_by_sector,
    rename_bio_resources,
    update_ampl_files,
    update_existing_infrastructure_metrics,
    techs_color_map,
    wood_list, wet_biomass_list, waste_list,
    LCA_DATA_FILES_DIR,
    AMPL_FILES_DIR,
    COMMON_DATA_DIR,
    REF_RESULTS,
    N_capita_2050,
    run_order_burden_shifts,
    default_colors_sankey,
    plot_sankey_carbon_flows,
    get_emissions_info,
    ssp_rcp_emissions_grouping,
    ssp_rcp_emissions_grouping_rev,
    unit_dict_plotly,
    sector_unit_dict,
)
from new_plots import _create_sankey_figure, generate_sankey_flows
from plots import *
import bw2data as bd
import pandas as pd
from mescal import Database
import ast
from tqdm import tqdm
import numpy as np
from plotly.subplots import make_subplots

In [ ]:
path_lca_data_2050 = AMPL_FILES_DIR / 'data' / '2050'
results_folder = 'multi_iam'

In [ ]:
impact_abbrev = pd.read_csv(COMMON_DATA_DIR / 'impact_abbrev.csv')
impact_abbrev.Impact_category = impact_abbrev.Impact_category.apply(lambda x: ast.literal_eval(x))
energyscope_model = pd.read_csv(COMMON_DATA_DIR / 'model_2050.csv')
es_tech_df = pd.read_csv(COMMON_DATA_DIR / 'technology_dictionary.csv')

In [ ]:
es_tech_df = es_tech_df[~es_tech_df['Programming name'].isin(wood_list+wet_biomass_list+waste_list)]  # keeping ES names for aggregation of biomass resources later on
es_tech_name_dict = dict(zip(es_tech_df['Programming name'], es_tech_df['Long name']))

## Define scenarios and run the model

In [ ]:
policies_scenarios = {
    # 'CCA REQ': {'limit_rhh': False, 'limit_req': True, 'limit_acc': True},
    # 'CCA RHH': {'limit_rhh': True, 'limit_req': False, 'limit_acc': True},
    # 'CCA': {'limit_rhh': False, 'limit_req': False, 'limit_acc': True},
    # 'RHH REQ': {'limit_rhh': True, 'limit_req': True, 'limit_acc': False},
    # 'REQ': {'limit_rhh': False, 'limit_req': True, 'limit_acc': False},
    # 'RHH': {'limit_rhh': True, 'limit_req': False, 'limit_acc': False},
    'None': {'limit_rhh': False, 'limit_req': False, 'limit_acc': False},
    'All': {'limit_rhh': True, 'limit_req': True, 'limit_acc': True},
}

In [ ]:
background_scenarios_list = [
    {"model": "image", "pathway": "SSP1-L", "year": 2050}, # +1.7°C
    {"model": "image", "pathway": "SSP2-M", "year": 2050}, # +2.9°C
    {"model": "image", "pathway": "SSP3-H", "year": 2050}, # +3.6°C
    {"model": "remind", "pathway": "SSP2-PkBudg1000", "year": 2050}, # +1.8°C
    {"model": "remind", "pathway": "SSP2-NPi", "year": 2050}, # +2.6°C
    {"model": "remind", "pathway": "SSP3-rollBack", "year": 2050}, # +3.5°C
    {"model": "tiam-ucl", "pathway": "SSP2-RCP26", "year": 2050}, # +1.8°C
    {"model": "tiam-ucl", "pathway": "SSP2-RCP45", "year": 2050}, # +2.8°C
    {"model": "tiam-ucl", "pathway": "SSP2-Base", "year": 2050}, # +3.1°C
    {"model": "message", "pathway": "SSP1-L", "year": 2050}, # +1.6°C
    {"model": "message", "pathway": "SSP2-M", "year": 2050}, # +2.9°C
    {"model": "message", "pathway": "SSP3-H", "year": 2050}, # +3.2°C
]

In [ ]:
multi_iam_grouping = [
    'all',  # constraints must hold for every background scenario
    'ssp_rcp',  # constraints must hold for the 4 IAM of each SSP-RCP (high, medium, low)
    'iam',  # constraints must hold for the 3 SSP-RCP of each IAM
    'individual',  # constraints for every IAM/SSP-RCP individually
]

In [ ]:
background_scenarios_group_dict = {
    'all': {'all': background_scenarios_list},
    'iam': {iam: [i for i in background_scenarios_list if i['model'] == iam] for iam in list(set([i['model'] for i in background_scenarios_list]))},
    'ssp_rcp': {ssp_rcp_emissions_grouping_rev[ssp_rcp]: [i for i in background_scenarios_list if ssp_rcp_emissions_grouping_rev[i['pathway']] == ssp_rcp_emissions_grouping_rev[ssp_rcp]] for ssp_rcp in list(set([i['pathway'] for i in background_scenarios_list]))},
    'individual': {f"{iam}-{ssp_rcp_emissions_grouping_rev[ssp_rcp]}": [i for i in background_scenarios_list if (i['model'] == iam) & (ssp_rcp_emissions_grouping_rev[i['pathway']] == ssp_rcp_emissions_grouping_rev[ssp_rcp])] for iam in list(set([i['model'] for i in background_scenarios_list])) for ssp_rcp in list(set([i['pathway'] for i in background_scenarios_list]))},
}

In [ ]:
obj_list = [
    'TotalCost',
    # 'TotalLCIA_REQD',
    # 'TotalLCIA_RHHD',
    # 'TotalABROAD_m_CCS_all',
]

In [ ]:
ccs_limit_list = [
    '10',
    '20',
    '30',
    '40',
    'Infinity',
]

In [ ]:
territorial_obj_list = [
    '-12',
    '-9',
    '-6',
    '-3',
    '0',
]

In [ ]:
bd.projects.set_current('ecoinvent3.12')

In [ ]:
if update_dat_files:

    df_co2_layers = pd.DataFrame(data=[], columns=['Name', 'Flow', 'Amount', 'amount', 'IAM', 'SSP-RCP'])

    for bg_scen in background_scenarios_list:
        db = Database([
            f'ecoinvent_cutoff_3.12_{bg_scen["model"]}_{bg_scen["pathway"]}_2050 - regionalized',
            f'EnergyScope_CA-QC_{bg_scen["model"]}_{bg_scen["pathway"]}_2050',
        ])
        df = update_ampl_files(
            iam_scenarios = [bg_scen],
            year = 2050,
            specific_lcia_abbrev = ['RHHD', 'REQD', 'm_CCS_all'],
            main_database = db,
            direct_emissions_files = False,
            territorial_emissions_files = True,
            ecoinvent_version = '3.12',
            iw_version = '2.2.1',
            return_co2_layers_df = True,
        )

        df['IAM'] = bg_scen["model"]
        df['SSP-RCP'] = bg_scen["pathway"]
        df_co2_layers = pd.concat([df_co2_layers, df])

    # CO2 layers are aligned for the endogenous estimation of carbon capture potentiel
    # Differences among IAM-SSP-RCP scenarios can occur
    # Therefore, for runs where all scenarios are satisfied at once, we align metrics with the minimum values among IAM-SSP-RCP scenarios
    # in order to avoid having some scenarios with higher carbon capture than concentrated emissions
    # Overall, this potentially leads to a slight under-estimation of the carbon capture potential for some scenarios
    df_co2_layers_min = df_co2_layers.drop(columns='Amount').groupby(['Name', 'Flow']).min().reset_index()
    with open(f'{path_lca_data_2050}/QC_lyrios_CO2.dat', 'w') as f:
        for index, row in df_co2_layers_min.iterrows():
            f.write(f"let layers_in_out['YEAR_2050','{row['Name']}','{row['Flow']}'] := {row['amount']} ;\n")

In [ ]:
run_list = [(grouping, key) for grouping in multi_iam_grouping for key in background_scenarios_group_dict[grouping].keys()]
run_list = [(policy, run[0], run[1]) for policy in policies_scenarios.keys() for run in run_list]
run_list = [(territorial_obj, run[0], run[1], run[2]) for territorial_obj in list(reversed(territorial_obj_list)) for run in run_list]
run_list = [(ccs_limit, run[0], run[1], run[2], run[3]) for ccs_limit in list(reversed(ccs_limit_list)) for run in run_list]
run_list = [(obj, run[0], run[1], run[2], run[3], run[4]) for obj in obj_list for run in run_list]

In [ ]:
len(run_list)

In [ ]:
results_dict = {}
infeasible_runs = []

if run_model:

    for run in run_list:

        (obj, ccs_limit, territorial_obj, burden_shifting_policy, grouping, key) = run
        print(run)

        if obj not in results_dict:
            results_dict[obj] = {}
        if ccs_limit not in results_dict[obj]:
            results_dict[obj][ccs_limit] = {}
        if territorial_obj not in results_dict[obj][ccs_limit]:
            results_dict[obj][ccs_limit][territorial_obj] = {}
        if burden_shifting_policy not in results_dict[obj][ccs_limit][territorial_obj]:
            results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy] = {}
        if grouping not in results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy]:
            results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy][grouping] = {}
        if key not in results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy][grouping]:
            results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy][grouping][key] = {}

        with open(AMPL_FILES_DIR / 'model' / 'QC_objective_function.mod', 'w') as f:
            f.write(f'drop obj;\nminimize obj2: sum{{y in YEARS}} {obj}[y];')


        # If a less constraining problem is infeasible, then this one is too
        territorial_obj_idx = territorial_obj_list.index(territorial_obj)
        ccs_limit_idx = ccs_limit_list.index(ccs_limit)
        if territorial_obj_idx + 1 < len(territorial_obj_list):
            if results_dict[obj][ccs_limit][territorial_obj_list[territorial_obj_idx+1]][burden_shifting_policy][grouping][key]['results'] is None:
                results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy][grouping][key]['results'] = None
                infeasible_runs.append(run)
                continue
        if ccs_limit_idx + 1 < len(ccs_limit_list):
            if results_dict[obj][ccs_limit_list[ccs_limit_idx+1]][territorial_obj][burden_shifting_policy][grouping][key]['results'] is None:
                results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy][grouping][key]['results'] = None
                infeasible_runs.append(run)
                continue
        if burden_shifting_policy == 'All':
            if results_dict[obj][ccs_limit][territorial_obj]['None'][grouping][key]['results'] is None:
                results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy][grouping][key]['results'] = None
                infeasible_runs.append(run)
                continue

        try:
            results = run_opti_multi_iam(
                iam_scenarios=background_scenarios_group_dict[grouping][key],
                constraint_on_remaining_hh=policies_scenarios[burden_shifting_policy]['limit_rhh'],
                constraint_on_remaining_eq=policies_scenarios[burden_shifting_policy]['limit_req'],
                constraint_on_foreign_ghg_emissions=policies_scenarios[burden_shifting_policy]['limit_acc'],
                constraint_on_territorial_ghg_emissions=territorial_obj,
                constraint_on_ccs=ccs_limit,
            )

            results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy][grouping][key]['results'] = results
            results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy][grouping][key]['TotalCost'] = results.variables['TotalCost']['TotalCost'].iloc[0]

            if save_results:
                with open(f"../03_Results/Pickle/{results_folder}/results_{obj}_{territorial_obj}_{ccs_limit}_{grouping}_{key}_{burden_shifting_policy}.pkl", 'wb') as f:
                    pickle.dump(results, f)

        except ValueError:
            results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy][grouping][key]['results'] = None
            infeasible_runs.append(run)

else:
    # To skip the optimization part
    for run in run_list:

        (obj, ccs_limit, territorial_obj, burden_shifting_policy, grouping, key) = run

        if obj not in results_dict:
            results_dict[obj] = {}
        if ccs_limit not in results_dict[obj]:
            results_dict[obj][ccs_limit] = {}
        if territorial_obj not in results_dict[obj][ccs_limit]:
            results_dict[obj][ccs_limit][territorial_obj] = {}
        if burden_shifting_policy not in results_dict[obj][ccs_limit][territorial_obj]:
            results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy] = {}
        if grouping not in results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy]:
            results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy][grouping] = {}
        if key not in results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy][grouping]:
            results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy][grouping][key] = {}

        try:
            with open(f"../03_Results/Pickle/{results_folder}/results_{obj}_{territorial_obj}_{ccs_limit}_{grouping}_{key}_{burden_shifting_policy}.pkl", 'rb') as f:
                results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy][grouping][key]['results'] = pickle.load(f)
        except FileNotFoundError:
            results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy][grouping][key]['results'] = None
            infeasible_runs.append(run)

In [ ]:
infeasible_runs

## Extract results

In [ ]:
impact_categories_list = [
    i for i in bd.methods if
    (i[0] == 'IMPACT World+ Damage 2.2.1_regionalized for ecoinvent v3.12')
    | (i[0] == 'IMPACT World+ Damage 2.2.1 for ecoinvent v3.12 (incl. CO2 uptake)')
]

impact_categories_list +=[
    ('IMPACT World+ Damage 2.2.1_regionalized for ecoinvent v3.12', 'Ecosystem quality', 'Remaining ecosystem quality'),
    ('IMPACT World+ Damage 2.2.1_regionalized for ecoinvent v3.12', 'Human health', 'Remaining human health'),
    ('IMPACT World+ Damage 2.2.1_regionalized for ecoinvent v3.12', 'Ecosystem quality', 'Total ecosystem quality (biogenic)'),
    ('IMPACT World+ Damage 2.2.1_regionalized for ecoinvent v3.12', 'Human health', 'Total human health (biogenic)'),
    ('IMPACT World+ Midpoint 2.2.1 for ecoinvent v3.12 (incl. CO2 uptake)', 'Midpoint', 'Climate change, short term, total'),
]

impact_categories_list = [i for i in impact_categories_list if i[2] not in ['Total ecosystem quality', 'Total human health']]
impact_categories_list = [i for i in impact_categories_list if  # keep only climate change and marine acidification from -1/+1 version of IW+
    not (i[0] == 'IMPACT World+ Damage 2.2.1_regionalized for ecoinvent v3.12' and ('Marine acidification' in i[2] or 'Climate change' in i[2]))
]

impact_categories_list = [i for i in impact_categories_list if not ('Climate change' in i[2] and 'total' not in i[2])]  # keep only climate change total

In [ ]:
imp_cat_var_dict = {
    'Remaining ecosystem quality': ['TotalLCIA', 'REQD'],
    'Remaining human health': ['TotalLCIA', 'RHHD'],
    'Climate change, short term, total (territorial)': ['TotalTERRITORIAL', 'm_CCS_all'],
    'Climate change, short term, total (abroad)': ['TotalABROAD', 'm_CCS_all'],
    'Climate change, short term, total': ['TotalLCIA', 'm_CCS_all'],
}

In [ ]:
limits = {}
for bg_scenario in background_scenarios_list:

    model = bg_scenario['model']
    pathway = bg_scenario['pathway']
    index_model = model.replace('-', '_').upper()
    index_pathway = ssp_rcp_emissions_grouping_rev[pathway].upper()

    norm_factors = pd.read_csv(path_lca_data_2050 / model / pathway / 'QC_techs_lca_max.csv')

    if model not in limits:
        limits[model] = {}

    limits[model][pathway] = {
        'RHH': results_dict['TotalCost']['Infinity']['0']['All']['all']['all']['results'].parameters['limit_lcia'].loc[index_model,index_pathway,'RHHD']['limit_lcia'],
        'REQ': results_dict['TotalCost']['Infinity']['0']['All']['all']['all']['results'].parameters['limit_lcia'].loc[index_model,index_pathway,'REQD']['limit_lcia'],
        'CCA': results_dict['TotalCost']['Infinity']['0']['All']['all']['all']['results'].parameters['limit_abroad'].loc[index_model,index_pathway,'m_CCS_all']['limit_abroad'],
    }

    limits[model][pathway]['CCA'] *= norm_factors.set_index('Abbrev').loc['m_CCS_all']['max_unit']
    limits[model][pathway]['RHH'] *= norm_factors.set_index('Abbrev').loc['RHHD']['max_unit']
    limits[model][pathway]['REQ'] *= norm_factors.set_index('Abbrev').loc['REQD']['max_unit']

In [ ]:
scenarios = [(bg_scenario['model'], bg_scenario['pathway']) for bg_scenario in background_scenarios_list]
scenarios = [(policy, scenario[0], scenario[1]) for policy in policies_scenarios.keys() for scenario in scenarios]
scenarios = [(territorial_obj, scenario[0], scenario[1], scenario[2]) for territorial_obj in territorial_obj_list for scenario in scenarios]
scenarios = [(ccs_limit, scenario[0], scenario[1], scenario[2], scenario[3]) for ccs_limit in ccs_limit_list for scenario in scenarios]
scenarios = [(obj, scenario[0], scenario[1], scenario[2], scenario[3], scenario[4]) for obj in obj_list for scenario in scenarios]

In [ ]:
scenarios_multi_iam = []
for obj, ccs_limit, territorial_obj, burden_shifting_policy, model, pathway in scenarios:
    for grouping in multi_iam_grouping:
        if grouping == 'all':
            key = 'all'
        elif grouping == 'iam':
            key = model
        elif grouping == 'ssp_rcp':
            key = ssp_rcp_emissions_grouping_rev[pathway]
        else:  # grouping == 'individual'
            key = f"{model}-{ssp_rcp_emissions_grouping_rev[pathway]}"
        if (obj, ccs_limit, territorial_obj, burden_shifting_policy, grouping, key) not in infeasible_runs:
            scenarios_multi_iam.append((obj, ccs_limit, territorial_obj, burden_shifting_policy, grouping, key, model, pathway))

In [ ]:
impact_scores_2023 = pd.read_csv(LCA_DATA_FILES_DIR / '2023' / 'impact_scores.csv')
df_contrib_ccst_2023 = pd.read_csv(LCA_DATA_FILES_DIR / '2023' / 'contribution_analysis_all_processes_ccst.csv')

for scenario in tqdm(scenarios):

    (obj, ccs_limit, territorial_obj, burden_shifting_policy, model, pathway) = scenario

    index_model = model.replace('-', '_').upper()
    index_pathway = ssp_rcp_emissions_grouping_rev[pathway].upper()

    impact_scores = pd.read_csv(LCA_DATA_FILES_DIR / '2050' / model / pathway / 'impact_scores.csv')
    impact_scores_direct = pd.read_csv(LCA_DATA_FILES_DIR / '2050' / model / pathway / 'impact_scores_direct_emissions.csv')
    df_contrib_ccst = pd.read_csv(LCA_DATA_FILES_DIR / '2050' / model / pathway / 'contribution_analysis_all_processes_ccst.csv')

    impact_scores = update_existing_infrastructure_metrics(
        df_impact_2050=impact_scores,
        df_impact_2020=impact_scores_2023,
        list_existing_techs=['HYDRO_DAM', 'HYDRO_RIVER', 'WIND_ONSHORE'],
    )

    df_contrib_ccst = update_existing_infrastructure_metrics(
        df_impact_2050=df_contrib_ccst,
        df_impact_2020=df_contrib_ccst_2023,
        list_existing_techs=['HYDRO_DAM', 'HYDRO_RIVER', 'WIND_ONSHORE'],
        col_name='act_name',
        col_type='act_type',
    )

    for grouping in multi_iam_grouping:

        if grouping == 'all':
            key = 'all'
        elif grouping == 'iam':
            key = model
        elif grouping == 'ssp_rcp':
            key = ssp_rcp_emissions_grouping_rev[pathway]
        else:  # grouping == 'individual'
            key = f"{model}-{ssp_rcp_emissions_grouping_rev[pathway]}"
        res = results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy][grouping][key]

        if res['results'] is None:  # infeasible run
            continue

        df_f_mult, df_annual_prod, df_annual_res = get_impact_scores(
            impact_category=impact_categories_list,
            df_impact_scores=impact_scores,
            df_terr_abroad_ccst=df_contrib_ccst,
            df_results=res['results'],
        )

        df_annual_prod_direct = get_impact_scores(
            assessment_type='direct',
            impact_category=impact_categories_list,
            df_impact_scores=impact_scores_direct,
            df_results=res['results'],
        )

        df_f_mult = df_f_mult[df_f_mult.F_Mult != 0]
        df_annual_prod = df_annual_prod[df_annual_prod.Annual_Prod != 0]
        df_annual_prod_direct = df_annual_prod_direct[df_annual_prod_direct.Annual_Prod != 0]
        df_annual_res = df_annual_res[df_annual_res.Annual_Res != 0]

        df_f_mult['Objective function'] = obj
        df_annual_prod['Objective function'] = obj
        df_annual_prod_direct['Objective function'] = obj
        df_annual_res['Objective function'] = obj

        df_f_mult['Territorial obj'] = territorial_obj
        df_annual_prod['Territorial obj'] = territorial_obj
        df_annual_prod_direct['Territorial obj'] = territorial_obj
        df_annual_res['Territorial obj'] = territorial_obj

        df_f_mult['CCS limit'] = ccs_limit
        df_annual_prod['CCS limit'] = ccs_limit
        df_annual_prod_direct['CCS limit'] = ccs_limit
        df_annual_res['CCS limit'] = ccs_limit

        df_f_mult['IAM'] = model
        df_annual_prod['IAM'] = model
        df_annual_prod_direct['IAM'] = model
        df_annual_res['IAM'] = model

        df_f_mult['SSP-RCP'] = pathway
        df_annual_prod['SSP-RCP'] = pathway
        df_annual_prod_direct['SSP-RCP'] = pathway
        df_annual_res['SSP-RCP'] = pathway

        df_f_mult['Policy'] = burden_shifting_policy
        df_annual_prod['Policy'] = burden_shifting_policy
        df_annual_prod_direct['Policy'] = burden_shifting_policy
        df_annual_res['Policy'] = burden_shifting_policy

        df_f_mult['Grouping'] = grouping
        df_annual_prod['Grouping'] = grouping
        df_annual_prod_direct['Grouping'] = grouping
        df_annual_res['Grouping'] = grouping

        df_f_mult['Key'] = key
        df_annual_prod['Key'] = key
        df_annual_prod_direct['Key'] = key
        df_annual_res['Key'] = key

        df_f_mult = df_f_mult.merge(
            energyscope_model[energyscope_model['Amount'] == 1],
            left_on='index',
            right_on='Name',
            how='left',
        ).rename(columns={'Flow': 'Main production'}).drop(columns=['Name', 'Amount'])

        df_annual_prod = df_annual_prod.merge(
            energyscope_model[energyscope_model['Amount'] == 1],
            left_on='index',
            right_on='Name',
            how='left',
        ).rename(columns={'Flow': 'Main production'}).drop(columns=['Name', 'Amount'])

        df_annual_prod_direct = df_annual_prod_direct.merge(
            energyscope_model[energyscope_model['Amount'] == 1],
            left_on='index',
            right_on='Name',
            how='left',
        ).rename(columns={'Flow': 'Main production'}).drop(columns=['Name', 'Amount'])

        df_f_mult['Main production'] = df_f_mult['Main production'].astype(str)
        df_annual_prod['Main production'] = df_annual_prod['Main production'].astype(str)
        df_annual_prod_direct['Main production'] = df_annual_prod_direct['Main production'].astype(str)

        df_annual_prod['Sector'] = df_annual_prod.apply(category_to_sector, axis=1)
        df_annual_prod_direct['Sector'] = df_annual_prod.apply(category_to_sector, axis=1)
        df_f_mult['Sector'] = df_f_mult.apply(category_to_sector, axis=1)
        df_annual_res['Sector'] = df_annual_res.apply(
            lambda x: 'Electricity' if x['index'] == 'ELECTRICITY_EHV'
            else ('Biomass' if x['index'] in wood_list+wet_biomass_list+waste_list else 'Imports'),
            axis=1)
        df_annual_res['Category'] = df_annual_res.apply(
            lambda x: 'ELECTRICITY_EHV' if x['index'] == 'ELECTRICITY_EHV'
            else ('Biomass' if x['index'] in wood_list+wet_biomass_list+waste_list else 'Imports'),
            axis=1)

        if model not in results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy][grouping][key]:
            results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy][grouping][key][model] = {}
        if pathway not in results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy][grouping][key][model]:
            results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy][grouping][key][model][pathway] = {}
        results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy][grouping][key][model][pathway]['df_f_mult'] = df_f_mult
        results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy][grouping][key][model][pathway]['df_annual_prod'] = df_annual_prod
        results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy][grouping][key][model][pathway]['df_annual_prod_direct'] = df_annual_prod_direct
        results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy][grouping][key][model][pathway]['df_annual_res'] = df_annual_res

In [ ]:
df_f_mult = pd.concat([results_dict[obj][ccs_limit][territorial_obj][policy_scenario][grouping][key][model][pathway]['df_f_mult'] for obj, ccs_limit, territorial_obj, policy_scenario, grouping, key, model, pathway in scenarios_multi_iam])
df_annual_prod = pd.concat([results_dict[obj][ccs_limit][territorial_obj][policy_scenario][grouping][key][model][pathway]['df_annual_prod'] for obj, ccs_limit, territorial_obj, policy_scenario, grouping, key, model, pathway in scenarios_multi_iam])
df_annual_prod_direct = pd.concat([results_dict[obj][ccs_limit][territorial_obj][policy_scenario][grouping][key][model][pathway]['df_annual_prod_direct'] for obj, ccs_limit, territorial_obj, policy_scenario, grouping, key, model, pathway in scenarios_multi_iam])
df_annual_res = pd.concat([results_dict[obj][ccs_limit][territorial_obj][policy_scenario][grouping][key][model][pathway]['df_annual_res'] for obj, ccs_limit, territorial_obj, policy_scenario, grouping, key, model, pathway in scenarios_multi_iam])

In [ ]:
cat_list = [i[-1] for i in impact_categories_list if
            i[1] in ['Human health', 'Ecosystem quality']
            and i[-1] not in ['Total human health', 'Total ecosystem quality']
            ] + ['Climate change, short term, total', 'Climate change, short term, total (territorial)', 'Climate change, short term, total (abroad)']

df_annual_prod_indirect = df_annual_prod.merge(
    df_annual_prod_direct,
    on=['Grouping', 'Key', 'Objective function', 'CCS limit', 'Territorial obj', 'Policy', 'IAM', 'SSP-RCP', 'index', 'Sector', 'Category', 'Annual_Prod'],
    suffixes=('', ' (direct)'),
    how='left',
)
for col in list(df_annual_prod.columns):
    if col not in ['Grouping', 'Key', 'Objective function', 'CCS limit', 'Territorial obj', 'Policy', 'IAM', 'SSP-RCP', 'index', 'Sector', 'Category', 'Annual_Prod', 'Phase', 'Main production']:
        if col in ['Climate change, short term, total (territorial)', 'Climate change, short term, total (abroad)']:
            df_annual_prod_direct[col] = 0  # phases are won't be used for these two categories
        else:
            df_annual_prod_indirect[col] = df_annual_prod_indirect[col] - df_annual_prod_indirect[f"{col} (direct)"]
            df_annual_prod_indirect.drop(columns=[f"{col} (direct)"], inplace=True)

df_annual_prod_indirect = df_annual_prod_indirect[['Grouping', 'Key', 'Objective function', 'CCS limit', 'Territorial obj', 'Policy', 'IAM', 'SSP-RCP', 'index', 'Sector', 'Category', 'Annual_Prod'] + cat_list]

In [ ]:
all_mob_techs = (
    list(results_dict['TotalCost']['Infinity']['0']['None']['all']['all']['results'].sets['TECHNOLOGIES_OF_FREIGHTMOB_ALL_DISTANCES']['TECHNOLOGIES_OF_FREIGHTMOB_ALL_DISTANCES'])
    + list(results_dict['TotalCost']['Infinity']['0']['None']['all']['all']['results'].sets['TECHNOLOGIES_OF_PRIVATEMOB_ALL_DISTANCES']['TECHNOLOGIES_OF_PRIVATEMOB_ALL_DISTANCES'])
    + list(results_dict['TotalCost']['Infinity']['0']['None']['all']['all']['results'].sets['TECHNOLOGIES_OF_PUBLICMOB_ALL_DISTANCES']['TECHNOLOGIES_OF_PUBLICMOB_ALL_DISTANCES'])
)

# Keeping mobility sub-models only for production and impact dataframe
df_annual_prod = df_annual_prod[~df_annual_prod['index'].isin(all_mob_techs)]
df_annual_prod_direct = df_annual_prod_direct[~df_annual_prod_direct['index'].isin(all_mob_techs)]
df_annual_prod_indirect = df_annual_prod_indirect[~df_annual_prod_indirect['index'].isin(all_mob_techs)]
df_f_mult_main_mob = df_f_mult[df_f_mult['index'].isin(all_mob_techs)]
df_f_mult = df_f_mult[~df_f_mult['index'].isin(all_mob_techs)]

df_annual_prod = aggregate_mobility_submodels(df_annual_prod)
df_annual_prod_direct = aggregate_mobility_submodels(df_annual_prod_direct)
df_annual_prod_indirect = aggregate_mobility_submodels(df_annual_prod_indirect)
df_f_mult = aggregate_mobility_submodels(df_f_mult)

df_annual_prod['index'] = df_annual_prod['index'].replace(es_tech_name_dict)
df_f_mult['index'] = df_f_mult['index'].replace(es_tech_name_dict)
df_f_mult_main_mob['index'] = df_f_mult_main_mob['index'].replace(es_tech_name_dict)
df_annual_res['index'] = df_annual_res['index'].replace(es_tech_name_dict)
df_annual_prod_direct['index'] = df_annual_prod_direct['index'].replace(es_tech_name_dict)
df_annual_prod_indirect['index'] = df_annual_prod_indirect['index'].replace(es_tech_name_dict)

In [ ]:
df_f_mult = pd.merge(
    df_f_mult,
    df_f_mult_main_mob[['Grouping', 'Key', 'Objective function', 'CCS limit', 'Territorial obj', 'Policy', 'IAM', 'SSP-RCP', 'index', 'C_inv_an', 'C_maint']],
    on=['Grouping', 'Key', 'Objective function', 'CCS limit', 'Territorial obj', 'Policy', 'IAM', 'SSP-RCP', 'index'],
    suffixes=('', '_main_mob'),
    how='left',
)

df_f_mult['C_inv_an_main_mob'] = df_f_mult['C_inv_an_main_mob'].fillna(0)
df_f_mult['C_maint_main_mob'] = df_f_mult['C_maint_main_mob'].fillna(0)
df_f_mult['C_inv_an'] += df_f_mult['C_inv_an_main_mob']
df_f_mult['C_maint'] += df_f_mult['C_maint_main_mob']
df_f_mult = df_f_mult.drop(columns=['C_inv_an_main_mob', 'C_maint_main_mob'])

In [ ]:
df_f_mult['Cost'] = df_f_mult['C_inv_an'] + df_f_mult['C_maint']
df_annual_res = df_annual_res.rename(columns={'C_op': 'Cost'})
df_annual_prod_direct['Cost'] = 0
df_annual_prod_indirect['Cost'] = 0

In [ ]:
col = ['Grouping', 'Key', 'Objective function', 'CCS limit', 'Territorial obj', 'Policy', 'IAM', 'SSP-RCP', 'index', 'Sector', 'Phase', 'Cost'] + cat_list
df_f_mult['Phase'] = 'Construction'
df_annual_prod['Phase'] = 'Operation'
df_annual_prod_direct['Phase'] = 'Operation (direct)'
df_annual_prod_indirect['Phase'] = 'Operation (indirect)'
df_annual_res['Phase'] = 'Resource'

df_total_impact = pd.concat([
    df_f_mult[['F_Mult'] + col].rename(columns={'F_Mult': 'Capacity or production'}),
    df_annual_prod_direct[['Annual_Prod'] + col].rename(columns={'Annual_Prod': 'Capacity or production'}),
    df_annual_prod_indirect[['Annual_Prod'] + col].rename(columns={'Annual_Prod': 'Capacity or production'}),
    df_annual_res[['Annual_Res'] + col].rename(columns={'Annual_Res': 'Capacity or production'}),
],
    ignore_index=True)

In [ ]:
path_results_tables = f"../03_Results/Tables/{results_folder}/"

In [ ]:
if save_results:
    df_f_mult.to_csv(path_results_tables+'df_f_mult.csv', index=False)
    df_annual_prod.to_csv(path_results_tables+'df_annual_prod.csv', index=False)
    df_annual_res.to_csv(path_results_tables+'df_annual_res.csv', index=False)
    df_annual_prod_direct.to_csv(path_results_tables+'df_annual_prod_direct.csv', index=False)
    df_total_impact.to_csv(path_results_tables+'df_total_impact.csv', index=False)

In [ ]:
# To skip previous parts (if results are already generated)
# df_f_mult = pd.read_csv(path_results_tables+'df_f_mult.csv')
# df_annual_prod = pd.read_csv(path_results_tables+'df_annual_prod.csv')
# df_annual_res = pd.read_csv(path_results_tables+'df_annual_res.csv')
# df_annual_prod_direct = pd.read_csv(path_results_tables+'df_annual_prod_direct.csv')
df_total_impact = pd.read_csv(path_results_tables+'df_total_impact.csv', na_values={'Policy': 'None'})

# Re-adjusting after reading
df_total_impact['Policy'] = df_total_impact['Policy'].replace({np.nan: 'None'})
if df_total_impact['CCS limit'].dtype == float or int:
    df_total_impact['CCS limit'] = df_total_impact['CCS limit'].apply(lambda x: str(int(x)) if x != np.Inf else 'Infinity')
if df_total_impact['Territorial obj'].dtype == float or int:
    df_total_impact['Territorial obj'] = df_total_impact['Territorial obj'].apply(lambda x: str(int(x)))

In [ ]:
# Pay attention that some runs were infeasible for 'All'/'all' but not the others
[i for i in infeasible_runs if i[3] == 'All' and (i[0], i[1], i[2], 'None', i[4], i[5]) not in infeasible_runs]

In [ ]:
total_impact_dict = df_total_impact.groupby(
    ['Grouping', 'Key', 'SSP-RCP', 'IAM', 'Policy', 'CCS limit', 'Territorial obj']
)[[
    'Remaining ecosystem quality',
    'Remaining human health',
    'Climate change, short term, total (abroad)',
    'Climate change, short term, total (territorial)',
]].sum().to_dict()

active_constraints = []

for scenario in scenarios_multi_iam:

    if scenario[:6] in infeasible_runs:
        continue

    (obj, ccs_limit, territorial_obj, burden_shifting_policy, grouping, key, model, pathway) = scenario

    if burden_shifting_policy == 'None':
        continue

    index_model = model.replace('-', '_').upper()
    index_pathway = ssp_rcp_emissions_grouping_rev[pathway].upper()
    norm_factors = pd.read_csv(path_lca_data_2050 / model / pathway / 'QC_techs_lca_max.csv')

    for cat in [
        'Remaining ecosystem quality',
        'Remaining human health',
        'Climate change, short term, total (abroad)',
        'Climate change, short term, total (territorial)',
    ]:

        # total_impact_from_df = total_impact_dict[cat][(grouping, key, pathway, model, burden_shifting_policy, ccs_limit, territorial_obj)]
        total_impact = results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy][grouping][key]['results'].variables[imp_cat_var_dict[cat][0]].loc[index_model,index_pathway,imp_cat_var_dict[cat][1]][imp_cat_var_dict[cat][0]] * norm_factors.set_index('Abbrev').loc[imp_cat_var_dict[cat][1]]['max_unit']

        if cat == 'Climate change, short term, total (territorial)':
            limit_impact = float(territorial_obj) * 1e3
        else:
            limit_impact = limits[model][pathway][imp_cat_code_dict[cat]]

        if limit_impact != 0:
            delta_limit = round(100 * (total_impact - limit_impact) / limit_impact, 3)
        else:
            delta_limit = round(total_impact - limit_impact, 5)

        active = (
                (imp_cat_code_dict[cat] in burden_shifting_policy or burden_shifting_policy == 'All')
                & (delta_limit == 0)
        )
        active_constraints.append([imp_cat_code_dict[cat], active, obj, grouping, key, model, pathway, burden_shifting_policy, ccs_limit, territorial_obj])

active_constraints = pd.DataFrame(
    data=active_constraints,
    columns=['Constraint', 'Active', 'Objective function', 'Grouping', 'Key', 'IAM', 'SSP-RCP', 'Policy', 'CCS limit', 'Territorial obj'],
)

In [ ]:
active_constraints.head()

In [ ]:
active_constraints[
    (active_constraints['Active'] == True)
    & (active_constraints['Grouping'] == 'all')
    & (active_constraints['Constraint'].isin(['RHH', 'REQ']))
]

## Plot the results

In [ ]:
import plotly.io as pio
import plotly.express as px
import plotly.graph_objs as go
pio.renderers.default = "png"

In [ ]:
df_total_impact['Climate change, short term, total'] *= 1e3 / N_capita_2050  # convert to t CO2 eq. per capita and per year
df_total_impact['Climate change, short term, total (abroad)'] *= 1e3 / N_capita_2050
df_total_impact['Climate change, short term, total (territorial)'] *= 1e3 / N_capita_2050

In [ ]:
df_total_impact['Total ecosystem quality (biogenic)'] *= 1e6 / N_capita_2050
df_total_impact['Remaining ecosystem quality'] *= 1e6 / N_capita_2050
df_total_impact['Total human health (biogenic)'] *= 1e6 / N_capita_2050
df_total_impact['Remaining human health'] *= 1e6 / N_capita_2050

In [ ]:
for bg_scenario in background_scenarios_list:
    limits[bg_scenario['model']][bg_scenario['pathway']]['REQ'] *= 1e6 / N_capita_2050
    limits[bg_scenario['model']][bg_scenario['pathway']]['RHH'] *= 1e6 / N_capita_2050
    limits[bg_scenario['model']][bg_scenario['pathway']]['CCA'] *= 1e3 / N_capita_2050

In [ ]:
df_total_impact['GHG emissions level'] = df_total_impact.apply(lambda x: ssp_rcp_emissions_grouping_rev[x['SSP-RCP']], axis=1)
df_total_impact = df_total_impact.rename(columns={'Policy': 'Burden shifting policy', 'Territorial obj': 'Carbon policy'})

In [ ]:
active_constraints['GHG emissions level'] = active_constraints.apply(lambda x: ssp_rcp_emissions_grouping_rev[x['SSP-RCP']], axis=1)
active_constraints = active_constraints.rename(columns={'Policy': 'Burden shifting policy', 'Territorial obj': 'Carbon policy'})

### Sankey diagrams of energy flows

In [ ]:
for run in run_list:
    (obj, ccs_limit, territorial_obj, burden_shifting_policy, grouping, key) = run
    res = results_dict[obj][ccs_limit][territorial_obj][burden_shifting_policy][grouping][key]['results']

    if res is None:  # infeasible run
        continue

    # if grouping != 'all':
    #     continue

    df_sankey = generate_sankey_flows(
        results=res,
        aggregate_mobility=True,
        aggregate_grid=True,
        aggregate_technology=True,
        run_id=0,
    )
    df_sankey['source (long)'] = df_sankey.apply(lambda x: es_tech_name_dict[x['source']] if x['source'] in es_tech_name_dict else x['source'], axis=1)
    df_sankey['target (long)'] = df_sankey.apply(lambda x: es_tech_name_dict[x['target']] if x['target'] in es_tech_name_dict else x['target'], axis=1)

    fig = _create_sankey_figure(df_sankey, colors=default_colors_sankey, long_names=True)

    if save_results:
        file_path = f'../03_Results/Figures/{results_folder}/{grouping}/'
        Path(file_path).mkdir(parents=True, exist_ok=True)
        fig.write_html(file_path+f'sankey_{obj}_{ccs_limit}_{territorial_obj}_{grouping}_{key}_{burden_shifting_policy}.html')

### Sankey of carbon flows

In [ ]:
for distance_level in ['_SD', '_MD', '_LD', '_ELD']:
    energyscope_model['Name'] = energyscope_model['Name'].str.replace(distance_level, '')
    energyscope_model['Flow'] = energyscope_model['Flow'].str.replace(distance_level, '')
energyscope_model.drop_duplicates(inplace=True)

energyscope_model['Name'] = energyscope_model['Name'].replace(es_tech_name_dict)

In [ ]:
for scenario in scenarios_multi_iam:

    (obj, ccs_limit, territorial_obj, burden_shifting_policy, grouping, key, model, pathway) = scenario

    # if grouping != 'all':
    #     continue

    df_plot = df_total_impact[
        (df_total_impact['Objective function'] == obj)
        & (df_total_impact['CCS limit'] == ccs_limit)
        & (df_total_impact['Carbon policy'] == territorial_obj)
        & (df_total_impact['Burden shifting policy'] == burden_shifting_policy)
        & (df_total_impact['IAM'] == model)
        & (df_total_impact['SSP-RCP'] == pathway)
        & (df_total_impact['Grouping'] == grouping)
        & (df_total_impact['Key'] == key)
    ].rename(columns={'Burden shifting policy': 'Run'})

    if len(df_plot) == 0:  # infeasible run
        continue

    fig = plot_sankey_carbon_flows(
        run=burden_shifting_policy,
        df_total_impact=df_plot,
        model=energyscope_model,
        cutoff=0,
        aggregate_technologies=True,
        show_figure=False,
        return_figure=True,
        save_results=False,
        per_capita=False,
        mode='mfa',
    )

    if save_results:
        file_path = f'../03_Results/Figures/{results_folder}/{grouping}/'
        fig.write_html(file_path+f'sankey_carbon_{obj}_{ccs_limit}_{territorial_obj}_{grouping}_{key}_{burden_shifting_policy}_{model}_{pathway}.html')

### Parallel coordinates

In [ ]:
df_config_std_mean = pd.merge(
    df_total_impact.groupby(['index', 'Phase'])[['Capacity or production']].std().reset_index(),
    df_total_impact.groupby(['index', 'Phase'])[['Capacity or production']].mean().reset_index(),
    how='outer',
    on=['index', 'Phase'],
    suffixes=(' (std)', ' (mean)')
)

df_config_std_mean = df_config_std_mean[
    (df_config_std_mean['Phase'].isin(['Resource', 'Operation (direct)']))
    & (df_config_std_mean['Capacity or production (mean)'] != 1e4)  # default value, useless technos
]

df_config_std_mean.head()

In [ ]:
plot_parallel_coord(df_total_impact, results_folder, save_fig=save_results)

### Box distributions

In [ ]:
fig = make_subplots(
    rows=3,
    cols=3,
    shared_xaxes=True,
    shared_yaxes=True,
    vertical_spacing=0.02,
    horizontal_spacing=0.02,
)

for i, y_axis in enumerate(['Natural gas', 'New Wind Energy Onshore', 'Cost']):
    for j, x_axis in enumerate(['CCS limit', 'Burden shifting policy', 'Carbon policy']):
        sub_fig_box, sub_fig_line, df_mean = plot_box_distribution_with_mean(
            df_total_impact, x_axis, y_axis
        )

        if sub_fig_line.layout.xaxis.categoryarray is not None:
            fig.update_xaxes(
                categoryorder='array',
                categoryarray=list(sub_fig_line.layout.xaxis.categoryarray),
                row=i + 1, col=j + 1,
            )

        if i == 2:
            fig.update_xaxes(title_text=x_axis, row=i+1, col=j+1)
        if j == 0:
            fig.update_yaxes(title_text=y_axis, row=i+1, col=j+1)

        for trace in sub_fig_box.data:
            fig.add_trace(trace, row=i+1, col=j+1)
        for trace in sub_fig_line.data:
            fig.add_trace(trace, row=i+1, col=j+1)

fig.update_layout(
    margin=dict(t=5, b=5, l=5, r=5),
)

if save_results:
    fig.write_html(f'../03_Results/Figures/{results_folder}/correlations.html')
    # fig.update_layout(
    #     height=1300,
    #     width=1200,
    # )
    # fig.write_image(f'../03_Results/Figures/{results_folder}/correlations.pdf')

### Contributions of impact categories

In [ ]:
plot_contrib_aop(df_total_impact, impact_categories_list, results_folder, 'Human health', save_fig=save_results)
plot_contrib_aop(df_total_impact, impact_categories_list, results_folder, 'Ecosystem quality', cutoff=0.1, save_fig=save_results)

### Active constraints regarding burden shifting

In [ ]:
for dim in ['CCS limit', 'GHG emissions level', 'IAM', 'Carbon policy']:
    plot_active_burden_shifting_constraints(active_constraints, df_total_impact, results_folder, dim, save_fig=True)

### Invariants and variants in the configurations

In [ ]:
plot_utilization_frequency_and_distribution(
    df_total_impact,
    results_folder,
    'Burden shifting policy',
    ['Electricity', 'Heat', 'Resource'],
    file_name='config_frequency_twh',
    save_fig=True,
)

In [ ]:
plot_utilization_frequency_and_distribution(
    df_total_impact,
    results_folder,
    'Burden shifting policy',
    ['Freight mobility', 'Passenger mobility'],
    file_name='config_frequency_mobility',
    save_fig=True,
)

In [ ]:
plot_utilization_frequency_and_distribution(
    df_total_impact,
    results_folder,
    'Burden shifting policy',
    ['Alternative fuels', 'Carbon storage'],
    file_name='config_frequency_other',
    save_fig=True,
)

### Correlation matrix

In [ ]:
df_corr = df_total_impact[df_total_impact['Phase'].isin(['Construction', 'Resource'])][['Grouping', 'Key', 'Carbon policy', 'Burden shifting policy', 'CCS limit', 'index', 'Capacity or production']].drop_duplicates().pivot(
    columns='index',
    index=['Grouping', 'Key', 'Carbon policy', 'Burden shifting policy', 'CCS limit'],
    values='Capacity or production',
).fillna(0)
df_corr = df_corr.drop(columns=[i for i in df_corr.columns if len(df_corr[i].unique()) == 1])  # remove constants
# df_corr = df_corr.drop(columns=[i for i in df_corr.columns if df_corr[i].max() < 1])  # major technos only
df_corr = df_corr.corr().dropna(how="all").dropna(how="all", axis=1)
df_corr.shape

In [ ]:
df_corr['Decentralized Heat Pump Electricity'].sort_values()

In [ ]:
fig = px.imshow(df_corr)
fig.write_html(f'../03_Results/Figures/{results_folder}/heatmap_corr.html')

### Cost of robustness

In [ ]:
dim = 'Cost'
groupby_dims = ['Grouping', 'Key', 'Burden shifting policy', 'CCS limit', 'Carbon policy']
df_delta_bs = plot_delta(df_total_impact, results_folder, dim, relative=True, return_fig=False, return_df=True)
df_delta_all = plot_delta(df_total_impact, results_folder, dim, delta_dim='Grouping', relative=True, return_fig=False, return_df=True)
df_delta_min_cost = df_total_impact.copy(deep=True)
df_delta_min_cost = df_delta_min_cost[groupby_dims + [dim]].drop_duplicates()
df_delta_min_cost = df_delta_min_cost.groupby(groupby_dims)[[dim]].sum().reset_index()
min_cost = df_delta_min_cost[dim].min()
df_delta_min_cost['Delta_rel'] = df_delta_min_cost[dim].apply(lambda x: np.log10(100 * (x - min_cost) / min_cost) if x != min_cost else -np.inf)

In [ ]:
dict_df_imp = {}
groupby_dims = ['IAM', 'SSP-RCP', 'Grouping', 'Key', 'Burden shifting policy', 'CCS limit', 'Carbon policy']
for cat in ['Remaining human health', 'Remaining ecosystem quality', 'Climate change, short term, total (abroad)', 'Climate change, short term, total (territorial)']:
    df = df_total_impact[df_total_impact['Grouping'] == 'all'].copy(deep=True)
    df = df[groupby_dims + [cat]].drop_duplicates()
    df = df.groupby(groupby_dims)[[cat]].sum().reset_index()
    min_ = df[cat].min()
    df['Delta_rel_min'] = df[cat].apply(lambda x: 100 * (x - min_) / min_)
    df = df.rename(columns={cat: "imp_cat"})
    df_active_constr = active_constraints[(active_constraints['Grouping'] == 'all') & (active_constraints['Constraint'] == imp_cat_code_dict[cat])]
    df = df.merge(
        df_active_constr,
        on=groupby_dims,
        how='left',
    )
    df['Key'] = df.apply(lambda x: f"{x['IAM']}-{ssp_rcp_emissions_grouping_rev[x['SSP-RCP']]}", axis=1)
    dict_df_imp[cat] = df

In [ ]:
plot_relative_cost_difference_heatmap(df=df_delta_bs, difference_type='bs', save_fig=save_results, file_name='heatmap_bs_constraints', results_folder=results_folder)
plot_relative_cost_difference_heatmap(df=df_delta_all, difference_type='grouping', save_fig=save_results, file_name='heatmap_grouping', results_folder=results_folder)
plot_relative_cost_difference_heatmap(df=df_delta_min_cost, difference_type='min_cost', save_fig=save_results, file_name='heatmap_min_cost', results_folder=results_folder)
for cat, df in dict_df_imp.items():
    plot_relative_cost_difference_heatmap(df=df, difference_type=cat, save_fig=save_results, file_name=f'heatmap_min_imp_{imp_cat_code_dict[cat].lower()}', results_folder=results_folder)

In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.02)
fig1 = plot_box_rel_delta(df=df_delta_bs, cat='bs', return_fig=True, save_fig=False)
fig2 = plot_box_rel_delta(df=df_delta_all, cat='all', return_fig=True, save_fig=False)

def is_num(name):
    try:
        float(name)
        return True
    except (TypeError, ValueError):
        return False

# Split fig1 traces: numeric names (territorial_obj) vs. symbols (SCEN)
num_traces = sorted(
    [t for t in fig1.data if is_num(t.name)],
    key=lambda t: float(t.name),
    reverse=True,            # 0.0, -3.0, -6.0, -9.0, -12.0 (set False for -12 ... 0)
)
sym_traces = [t for t in fig1.data if not is_num(t.name)]

# Assign each group to its own legend
for t in num_traces:
    t.legend = 'legend'
for t in sym_traces:
    t.legend = 'legend2'

# Add to the main figure: numbers first so the legend order follows
for trace in num_traces + sym_traces:
    fig.add_trace(trace, row=1, col=1)

for trace in fig2.data:
    trace.showlegend = False
    fig.add_trace(trace, row=2, col=1)

fig.update_xaxes(
    categoryorder=fig1.layout.xaxis.categoryorder,
    categoryarray=fig1.layout.xaxis.categoryarray,
    row=1, col=1,
)
fig.update_xaxes(
    title_text='(<i>limit<sub>CCS</sub></i>, <i>territorial<sub>obj</sub></i>)',
    categoryorder=fig2.layout.xaxis.categoryorder,
    categoryarray=fig2.layout.xaxis.categoryarray,
    row=2, col=1,
)
fig.update_yaxes(type='log', title_text=r'$\Delta_{rel}^{y_{BS}}{C_{tot}} \ \text{(%)}$', row=1, col=1)
fig.update_yaxes(type='log', title_text=r'$\Delta_{rel}^{all}{C_{tot}} \ \text{(%)}$', row=2, col=1)
fig.update_layout(
    legend=dict(
        title=dict(text='<i>territorial<sub>obj</sub></i>', side='left'),
        orientation='h',
        x=0, xanchor='left',
        y=1.02, yanchor='bottom',
    ),
    legend2=dict(
        title=dict(text='<i>SCEN</i>', side='left'),
        orientation='h',
        x=0, xanchor='left',
        y=1.07, yanchor='bottom',
    ),
    margin=dict(t=5, b=5, l=5, r=5),
)

if save_results:
    fig.write_html(f'../03_Results/Figures/{results_folder}/box_delta_rel_cost.html', include_mathjax="cdn")
    # fig.update_layout(height=750, width=900)
    # fig.write_image(f'../03_Results/Figures/{results_folder}/box_delta_rel_cost.pdf')

In [ ]:
for cat, df in dict_df_imp.items():
    plot_box_rel_delta(df=df, cat=cat, save_fig=save_results, results_folder=results_folder)

### Contributions of technologies/sectors

In [ ]:
def plot_contribution_by_sector_burden(
        model,
        pathway,
        impact_category,
        contribution_of: str = 'tech',
        save_fig: bool = save_results,
        show_fig: bool = False,
):

    fig = plot_contribution_by_sector(
        df=df_total_impact[
            (df_total_impact['IAM'] == model)
            & (df_total_impact['SSP-RCP'] == pathway)
        ].rename(columns={
            'Policy': 'Run',
            'Total human health (biogenic)': 'Total human health',
            'Total ecosystem quality (biogenic)': 'Total ecosystem quality',
        }),
        imp_cat=impact_category,
        save_results=False,
        group_by='index' if contribution_of == 'tech' else 'Sector',
        cutoff=0.025 if contribution_of == 'tech' else 0,
        showlegend=True,
        show_direct_marker=False,
        show_total_marker=True,
        show_regionalized_marker=False,
        hatch_phase=False,
        multiple_rcp=False,
        study='burden',
        x_label_name='Scenario',
        return_fig=True,
        show_fig=False,
    )

    if impact_category in [
        'Remaining ecosystem quality',
        'Remaining human health',
        'Climate change, short term, total (abroad)',
    ]:

        fig.add_hline(
            y=limits[model][pathway][imp_cat_code_dict[impact_category]],
            line_width=3,
            line_dash="dash",
            line_color='red',
            annotation_text='No burden shift',
            annotation=dict(font_color='red'),
        )

    if show_fig:
        fig.show()

    if save_fig:
        fig.write_html(f'../03_Results/Figures/{results_folder}/{model}/{pathway}/contrib_{contribution_of}_{model}_{pathway}_{imp_cat_code_dict[impact_category].lower()}.html')

In [ ]:
for cat in [
    'Total ecosystem quality',
    'Total human health',
    'Remaining ecosystem quality',
    'Remaining human health',
    'Climate change, short term, total (abroad)',
    'Climate change, short term, total (territorial)',
    'Climate change, short term, total',
]:
    for contrib_type in ['tech', 'sector']:
        for bg_scenario in background_scenarios_list:
            plot_contribution_by_sector_burden(
                model=bg_scenario['model'],
                pathway=bg_scenario['pathway'],
                impact_category=cat,
                contribution_of=contrib_type,
            )

In [ ]:
plot_contribution_by_sector_burden(
    model='tiam-ucl',
    pathway='SSP2-RCP26',
    impact_category='Remaining human health',
    show_fig=True,
)

### Distribution of impacts across IAMs and SSP-RCP scenarios

In [ ]:
def plot_impact_distribution(impact_category, show_fig: bool=True, save_fig: bool=save_results):

    df_plot = df_total_impact.groupby(['Objective function', 'Policy', 'IAM', 'SSP-RCP'])[[impact_category]].sum().reset_index()
    # df_plot['Constraint'] = df_plot.apply(lambda x: True if (imp_cat_code_dict[impact_category] in x['Policy'] or 'All' in x['Policy']) else False, axis=1)

    fig = px.box(
        df_plot,
        x='Objective function',
        y=impact_category,
        points='all',
        # color='Constraint',
        category_orders={'Policy': run_order_burden_shifts, 'Objective function': obj_list},
        hover_data=['Objective function', 'Policy', 'IAM', 'SSP-RCP'],
    )

    # fig.update_yaxes(range=[0, None])

    if show_fig:
        fig.show()

    if save_fig:
        fig.write_html(f'../03_Results/Figures/{results_folder}/distribution_{imp_cat_code_dict[impact_category]}.html')

In [ ]:
plot_impact_distribution('Remaining human health')

In [ ]:
plot_impact_distribution('Remaining ecosystem quality')

In [ ]:
plot_impact_distribution('Climate change, short term, total (abroad)')

### Influence of IAMs and SSP-RCP scenarios

In [ ]:
df_plot = df_total_impact.groupby(['IAM', 'GHG emissions level', 'Policy']).sum().reset_index()

In [ ]:
fig = px.imshow(
    df_plot.pivot_table(index='Policy', columns='IAM', values='Climate change, short term, total (abroad)', aggfunc='mean')
)

fig.show()

In [ ]:
fig = px.imshow(
    df_plot.pivot_table(index='Policy', columns='GHG emissions level', values='Climate change, short term, total (abroad)', aggfunc='mean')
)

fig.show()

In [ ]:
df_plot_pivot = df_plot.pivot(index='Policy', columns=['IAM', 'SSP-RCP category'], values='Climate change, short term, total (abroad)')
df_plot_pivot.columns = [f"{i[0]} - {i[1]}" for i in df_plot_pivot.columns]

fig = px.imshow(
    df_plot_pivot
)

fig.show()

### Cost

In [ ]:
tc_dict = {}
for obj, policy_scenario, model, pathway in scenarios:
    if obj not in tc_dict:
        tc_dict[obj] = {}
    if policy_scenario not in tc_dict[obj]:
        tc_dict[obj][policy_scenario] = {}
    if model not in tc_dict[obj][policy_scenario]:
        tc_dict[obj][policy_scenario][model] = {}
    tc_dict[obj][policy_scenario][model][pathway] = results_dict[obj][policy_scenario][model][pathway]['results'].variables['TotalCost']['TotalCost'].iloc[0]

In [ ]:
delta_rel_dict = {}
for obj, policy_scenario, model, pathway in scenarios:
    if obj not in delta_rel_dict:
        delta_rel_dict[obj] = {}
    if policy_scenario not in delta_rel_dict[obj]:
        delta_rel_dict[obj][policy_scenario] = {}
    if model not in delta_rel_dict[obj][policy_scenario]:
        delta_rel_dict[obj][policy_scenario][model] = {}

    delta_rel_dict[obj][policy_scenario][model][pathway] = 100 * (tc_dict[obj][policy_scenario][model][pathway] - tc_dict[obj]['None'][model][pathway]) / tc_dict[obj]['None'][model][pathway]

In [ ]:
tc_dict

In [ ]:
delta_rel_dict

In [ ]:
def plot_cost_difference(
        df_f_mult: pd.DataFrame,
        df_annual_res: pd.DataFrame,
        cutoff: float = 0,
        relative: bool = True,
        save_results: bool = save_results,
):

    df_f_mult['Cost'] = df_f_mult['C_inv_an'] + df_f_mult['C_maint']
    df_cost = pd.concat([df_f_mult[['Run', 'index', 'Cost']], df_annual_res[['Run', 'index', 'C_op']].rename(columns={'C_op': 'Cost'})])
    df_cost['Cost'] *= 1e6 / N_capita_2050  # from MCAD to CAD/cap

    # Ensure every index in the reference has an entry for every Run in df_cost
    for name in df_cost['index'].unique():
        for run in df_cost['Run'].unique():
            if run != 'None':  # Skip the default run
                if len(df_cost[(df_cost['index'] == name) & (df_cost['Run'] == run)]) == 0:
                    # Add a row with Production=0
                    new_row = {'index': name, 'Run': run, 'Cost': 0}
                    df_cost = pd.concat([df_cost, pd.DataFrame([new_row])], ignore_index=True)

    df_cost = pd.merge(
        df_cost,
        df_cost[df_cost['Run'] == 'None'][['index', 'Cost']],
        on=['index'],
        how='left',
        suffixes=('', '_ref'),
    )

    df_cost['Cost_ref'] = df_cost['Cost_ref'].fillna(0)
    df_cost['Delta'] = df_cost['Cost'] - df_cost['Cost_ref']
    df_cost['Delta_rel'] = 100 * df_cost['Delta'] / df_cost[df_cost['Run'] == 'None']['Cost'].sum()

    # Apply cutoff
    df_cost['index'] = df_cost.apply(lambda x: x['index'] if abs(x['Delta_rel']) >= cutoff else 'Other', axis=1)

    # Sort bars
    df_grouped_index = df_cost.groupby(['Run', 'index'])['Delta'].sum().reset_index()
    index_order = [
                      tec for tec in
                      df_grouped_index[df_grouped_index['Delta'] < 0].sort_values(by='Delta', ascending=False)['index'].unique().tolist() +
                      df_grouped_index[df_grouped_index['Delta'] > 0].sort_values(by='Delta', ascending=False)['index'].unique().tolist()
                      if tec != 'Other'
                  ] + ['Other']

    if relative:
        y_var = 'Delta_rel'
        y_label = 'Relative cost difference with respect to <i>None</i> scenario (%)'
    else:
        y_var = 'Delta'
        y_label = 'Cost difference with respect to <i>None</i> scenario (CAD/cap)'

    fig = px.bar(
        df_cost[df_cost['Run'] != 'None'].groupby(['Run', 'index'])[[y_var]].sum().reset_index(),
        y=y_var,
        x='Run',
        color='index',
        category_orders={'Run': run_order_burden_shifts, 'index': index_order},
        color_discrete_map=techs_color_map,
        labels={
            'Run': 'Scenario',
            y_var: y_label,
            'index': 'Technology or resource',
        }
    )

    fig.update_layout(legend_traceorder="reversed", margin=dict(t=5, b=5, l=5, r=5))

    x = df_cost[df_cost['Run'] != 'None'].groupby(['Run'])[[y_var]].sum().reset_index()['Run']
    y = df_cost[df_cost['Run'] != 'None'].groupby(['Run'])[[y_var]].sum().reset_index()[y_var]

    fig.add_trace(
        go.Scatter(
            x=x,
            y=y,
            mode="markers",
            marker=dict(color="red", size=8, symbol="circle", ),
            name="Net cost difference",
        )
    )

    fig.show()

    if save_results:
        fig.write_image(f'../03_Results/Figures/cost_difference_{'rel' if relative else 'abs'}.pdf')
        fig.update_layout(width=None, height=None)
        fig.write_html(
            f'../03_Results/Figures/cost_difference_{'rel' if relative else 'abs'}.html',
            include_plotlyjs=True,
            full_html=True,
        )

In [ ]:
plot_cost_difference(
    df_f_mult=df_f_mult,
    df_annual_res=df_annual_res,
    cutoff=1.0, # [%]
    relative=True,
    save_results=save_results,
)

In [ ]:
plot_cost_difference(
    df_f_mult=df_f_mult,
    df_annual_res=df_annual_res,
    cutoff=1.0, # [%]
    relative=False,
    save_results=save_results,
)